In [2]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score

Project_root=Path.cwd()
if Project_root.name=="notebooks":
    Project_root=Project_root.parent
sys.path.append(str(Project_root/"src"))

import data_loader, training, models, evaluation
for libs in (data_loader,training,models,evaluation):
    importlib.reload(libs)
from data_loader import load_subset_by_days, decode_datetimes, create_binary_label
from models import ANNModel
from training import train_model
from evaluation import predict_model

data_path=r"C:/Users/urvi_/Downloads/dataset.h5"
days=['2023-06-17','2023-06-18','2023-06-19']
x_clean, y_dist, dt_raw, ship_info, good_channels=load_subset_by_days(data_path,days)

x_all=np.log10(x_clean.max(axis=1)+1e-20).astype(np.float32)
y_all=create_binary_label(y_dist,threshold=1000)
day_of=np.array(decode_datetimes(dt_raw).date)

del x_clean

print(x_all.shape,y_all.shape,sorted(set(day_of)))


(25848, 100) (25848,) [datetime.date(2023, 6, 17), datetime.date(2023, 6, 18), datetime.date(2023, 6, 19)]


In [6]:
from typing import Any


class VesselDetection(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

days_sorted=sorted(set(day_of))
def run_fold(test_day,epochs=100,lr=1e-3,batch=64,dropout=0.3,wd=1e-4,seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    i=days_sorted.index(test_day)
    val_day=days_sorted[(i+1)%len(days_sorted)]
    test_idx=np.where(day_of==test_day)[0]
    tr_idx=np.where(day_of!=test_day)[0]
    cut=int(len(tr_idx)*0.8)
    fit_idx=np.where((day_of!=test_day)&(day_of!=val_day))[0]
    val_idx=np.where(day_of==val_day)[0]
    ss=StandardScaler().fit(x_all[fit_idx])
    mk=lambda idx:DataLoader(VesselDetection(ss.transform(x_all[idx]),y_all[idx]),
                             batch_size=batch,shuffle=(idx is fit_idx))
    train_loader,val_loader,test_loader=mk(fit_idx),mk(val_idx),mk(test_idx)
    n_far,n_near=(y_all[fit_idx]==0).sum(),(y_all[fit_idx]==1).sum()
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_far/n_near],dtype=torch.float32).to(device))
    model=ANNModel(input_size=x_all.shape[1],hidden1=256,hidden2=128,dropout=dropout).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=wd)
    history=train_model(model,train_loader,criterion,optimizer,device,epochs=epochs,val_loader=val_loader,patience=10)
    
    y_true,y_pred,y_prob=predict_model(model,test_loader,device)
    y_true,y_pred=y_true.astype(int),y_pred.astype(int).flatten()
    far_f1,near_f1=f1_score(y_true,y_pred,average=None,zero_division=0)
    
    best_epoch=int(np.argmin(history["val_loss"]))+1
    return {
        "test_day":str(test_day),
        "val_day":str(val_day),
        "best_epoch":best_epoch,
        "epochs_run":len(history["train_loss"]),
        "accuracy":accuracy_score(y_true,y_pred),
        "weighted_f1":f1_score(y_true,y_pred,average="weighted",zero_division=0),
        "far_f1":far_f1,
        "near_f1":near_f1,
        "auc":roc_auc_score(y_true,y_prob.flatten())
    },history
    
results,histories=[],{}
for d in sorted(set(day_of)):
    r,h=run_fold(d)
    results.append(r)
    histories[str(d)]=h
    print(r)

df=pd.DataFrame(results)
print(df.round(4))
num=df.drop(columns=["test_day","val_day"])
print(num.agg(["mean","std"]).round(4))

df["model"]="ANN"
df["split"]="leave_one_day_out_3days"
df.to_csv(Project_root/"results"/"tables"/"ann_daywise_results_3days.csv",index=False)


Epoch 1/100 | Train loss: 0.5456 | Val loss: 1.3700
Epoch 2/100 | Train loss: 0.4708 | Val loss: 1.6483
Epoch 3/100 | Train loss: 0.4377 | Val loss: 2.6081
Epoch 4/100 | Train loss: 0.4158 | Val loss: 0.9721
Epoch 5/100 | Train loss: 0.3845 | Val loss: 1.5077
Epoch 6/100 | Train loss: 0.3766 | Val loss: 1.3170
Epoch 7/100 | Train loss: 0.3568 | Val loss: 2.0671
Epoch 8/100 | Train loss: 0.3503 | Val loss: 1.4939
Epoch 9/100 | Train loss: 0.3251 | Val loss: 2.7582
Epoch 10/100 | Train loss: 0.3062 | Val loss: 1.2514
Epoch 11/100 | Train loss: 0.2952 | Val loss: 1.8751
Epoch 12/100 | Train loss: 0.2899 | Val loss: 1.4409
Epoch 13/100 | Train loss: 0.2698 | Val loss: 1.8778
Epoch 14/100 | Train loss: 0.2577 | Val loss: 1.9081
Early stopping at epoch 14
{'test_day': '2023-06-17', 'val_day': '2023-06-18', 'best_epoch': 4, 'epochs_run': 14, 'accuracy': 0.851439182915506, 'weighted_f1': 0.8527948227764897, 'far_f1': np.float64(0.8879159369527145), 'near_f1': np.float64(0.7797660013764625), 'a